# GermoVision-Net

End-to-end reproducible notebook for the pathogen drift-and-emergence forecasting pipeline.

**Model.** Causal dilated 1D-CNN -> BiLSTM -> Multi-head self-attention -> attention pooling -> five heads (probabilistic, event, Dirichlet policy, critic, reconstruction).

**Task.** Given a 42-day window of 12 features for a (region, lineage) pair, produce a 30-day probabilistic forecast of the logit share, an emergence-event probability, a per-region sequencing budget allocation, and a country arrival ETA.

**Training.** Three stages: (A) self-supervised masked reconstruction, (B) supervised NLL + focal, (C) PPO fine-tuning of the policy.

**Validation.** Rolling-origin walk-forward with a 30-day gap; no row shuffling.

**Single source of truth.** All constants, feature/metric/figure/table registries live in `germovision_config`; the pipeline exports `results.json`, `metrics_table.md`, `germovision_net.pt`, `germovision_net.onnx`, and 15 figures under `figures/`.

## 1. Environment and consistency check

In [ ]:
import json, math, random, time, logging, sys
import numpy as np, torch

import germovision_config as C
import germovision_data as D
import germovision_losses as L
import germovision_eval as E
import germovision_plots as P
from germovision_model import GermoVisionNet, ExportWrapper, count_flops
import germovision_train as T

T.setup_logging()
T.set_seed(C.SEED)

for line in C.assert_consistency():
    print(' OK ', line)

print()
print(f'Window T_IN = {C.T_IN} days, horizon H = {C.HORIZON} days, features = {C.N_FEATURES}')
print(f'Receptive field of causal CNN = {C.RECEPTIVE_FIELD} (must exceed the window)')

## 2. Load the panel

Try Nextstrain open metadata first; fall back to a synthetic panel with known true fitness advantages `s_i` — the fallback is required for verifiability (on real data `s_i` is unobservable).

In [ ]:
cfg = C.RunConfig(seed=C.SEED, fast=True, folds=2, use_real_data=False,
                  max_windows=500, main_max_windows=1500)
panel = D.load_panel(cfg)
print('Source :', panel.source)
print('Shape  :', panel.counts.shape, '(R, L, T)')
print('Regions:', ', '.join(panel.region_names))
print('True s :', np.round(panel.true_fitness, 3))

## 3. Rolling-origin folds and windowing

In [ ]:
folds = D.rolling_origin_folds(panel.n_days, k=cfg.folds, gap=C.GAP_DAYS)
for f in folds:
    print(f'fold {f.index}:  train_end={f.train_end:>3}  test=[{f.test_start:>3}, {f.test_end:>3})')

tr, va, te, raw_te, scaler = T.prepare_fold(panel, folds[-1], cfg)
print()
print(f'train windows: {len(tr):>5}   val: {len(va):>4}   test: {len(te):>4}')
print(f'input tensor  shape: {tr.x.shape}')
print(f'target tensor shape: {tr.y.shape}')

## 4. Build the model

In [ ]:
net = GermoVisionNet()
print(f'parameters : {net.n_parameters()/1e6:6.3f} M')
print(f'FLOPs      : {count_flops(net)/1e6:6.1f} M per forecast')
sample = torch.from_numpy(tr.x[:2])
with torch.no_grad():
    out = net(sample)
for k, v in out.items():
    print(f'  {k:12s} {tuple(v.shape)}')

## 5. Train stages A -> B

Stage A: masked reconstruction on 15% of the window in 3-7 day spans (single-point masks are trivially interpolated).

Stage B: NLL + 0.5 * focal loss with depth-based sample weights `w = n / (n + 50)`; early stopping on CRPS.

In [ ]:
t0 = time.perf_counter()
net, history_ab, stop_epoch = T.train_configuration(tr, va, cfg, seed=cfg.seed, fold=False)
print(f'training A+B done in {time.perf_counter()-t0:.1f} s; early stop epoch = {stop_epoch}')

## 6. Stage C: PPO fine-tuning of the sequencing-budget policy

In [ ]:
rng = np.random.default_rng(cfg.seed)
history_c, alloc, outbreak = T.stage_c_ppo(net, panel, scaler, cfg.epochs('C', fold=False), rng)
print(f'PPO epochs run: {len(history_c)}, final mean reward: {history_c[-1]["reward"]:+.3f}')
print(f'outbreak flagged: region={outbreak[0]}, day={outbreak[1]}')

## 7. Evaluate: our model + all baselines on the SAME test windows

Same-windows evaluation is required for the paired Diebold-Mariano test.

In [ ]:
res = T.evaluate_all(net, tr, te, raw_te, panel, cfg.seed, with_baselines=True)
for name, met in res['metrics'].items():
    flag = '*' if name == C.MODEL_NAME else ' '
    print(f'{flag} {name:22s}  MAE={met["MAE"]:.4f}  CRPS={met["CRPS"]:.4f}  '
          f'PICP80={met["PICP80"]:.3f}')

## 8. Full pipeline: folds + ablation + figures + tables

Runs the same protocol as `python germovision_train.py --fast --no-real-data --folds 2 --max-windows 500`, writes `results.json`, `metrics_table.md`, both checkpoints, and all 15 figures.

In [ ]:
T.main(['--fast', '--no-real-data', '--folds', '2', '--max-windows', '500'])

## 9. Inspect the results

In [ ]:
results = json.loads(C.RESULTS_JSON.read_text(encoding='utf-8'))
print('Data source :', results['data_source'])
print('Folds       :', results['n_folds'])
print('Elapsed     :', results['elapsed_sec'], 's')
print('ONNX exported:', results['onnx_exported'])
print()
print('GermoVision-Net main metrics (mean over folds):')
for spec in C.METRICS:
    v = results['metrics'][C.MODEL_NAME].get(spec.key)
    se = results['metrics_se'][C.MODEL_NAME].get(spec.key)
    print(f'  {spec.key:7s} {spec.format(v):>10s} +/- {spec.format(se):>8s}   {spec.name_en}')

## 10. Display selected figures inline

In [ ]:
from IPython.display import Image, display, Markdown
for n in (1, 4, 5, 8, 9, 10, 11, 12):
    spec = C.figure(n)
    display(Markdown(f'**{spec.full_caption}**'))
    display(Image(filename=str(spec.path_png)))

## 11. Load the exported model and run inference

In [ ]:
ckpt = torch.load(C.CHECKPOINT_PT, map_location='cpu', weights_only=False)
print('Checkpoint config:', ckpt['config'])

loaded = GermoVisionNet()
loaded.load_state_dict(ckpt['state_dict'])
loaded.eval()

with torch.no_grad():
    out = loaded(torch.from_numpy(te.x[:4]))
print('mu   :', out['mu'].shape,    '(batch, horizon)')
print('sigma:', out['sigma'].shape, '(batch, horizon)')
print('event prob:', torch.sigmoid(out['event_logit']).tolist())

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession(str(C.CHECKPOINT_ONNX), providers=['CPUExecutionProvider'])
out_onnx = sess.run(None, {'window': te.x[:4]})
for name, arr in zip(['mu', 'sigma', 'event_logit', 'alpha', 'attention'], out_onnx):
    print(f'{name:12s} {arr.shape}')

## 12. Print the three publication-ready tables

In [ ]:
from IPython.display import Markdown
Markdown(C.METRICS_TABLE_MD.read_text(encoding='utf-8'))